# Analyzing a Zeeschuimer export

Loads an Instagram NDJSON file captured with Zeeschuimer, then explores and graphs it with pandas.

To use a different capture, change `NDJSON_FILE` below.

In [ ]:
import base64
import html
import json
import os

import pandas as pd
from IPython.display import HTML
# makes sure graphs are displayed directly in the notebook
%matplotlib inline 

NDJSON_FILE = "../nov8posts.ndjson"  # change to analyze a different capture
IMAGE_FOLDER = "../nov8images"  # change to use a different folder of saved images
USE_INSTAGRAM_URLS = False  # True = Instagram's live links (these expire), False = saved images

## 1. Load the NDJSON as a dataframe

Each line of the file is one JSON object. `json_normalize` flattens the nested fields into columns with dotted names, such as `data.user.username`.

In [ ]:
with open(NDJSON_FILE, encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

df = pd.json_normalize(records)
df.shape

## 2. List all the available fields

In [ ]:
print(len(df.columns), "fields")
print()
print("\n".join(df.columns))

## 3. Keep only the fields we want

`rename` gives the columns shorter names. The dataframe is still in the order the posts were captured from the feed.

In [ ]:
columns = {
    "data.id": "id",
    "data.media_type": "media_type",
    "data.user.username": "username",
    "data.user.full_name": "display_name",
    "data.caption.text": "caption",
    "data.user.friendship_status.following": "following",
    "data.like_count": "likes",
    "data.comment_count": "comments",
    "data.is_paid_partnership": "paid_partnership",
    "data.taken_at": "date_posted",
}

posts = df[list(columns)].rename(columns=columns)

# Treat anything not explicitly True as False
posts["following"] = posts["following"] == True
posts["paid_partnership"] = posts["paid_partnership"] == True

# taken_at is stored in seconds since 1970 (UTC)
posts["date_posted"] = pd.to_datetime(posts["date_posted"], unit="s")

posts.head()

### Show the posts with image thumbnails

Pandas can't put images in a table by itself, so we turn the table into HTML, with an `<img>` tag in the thumbnail column. Saved images come from the folder made by `download_images.py`, and get embedded in the notebook so that it still displays them if you share it.

In [ ]:
def thumbnail(source):
    if pd.isna(source):
        return ""
    if USE_INSTAGRAM_URLS:
        return f'<img src="{source}" width="100">'
    if not os.path.exists(source):
        return ""
    with open(source, "rb") as f:
        encoded = base64.b64encode(f.read()).decode()
    return f'<img src="data:image/jpeg;base64,{encoded}" width="100">'


if USE_INSTAGRAM_URLS:
    sources = df["data.image_versions2.candidates"].str[0].str["url"]
else:
    sources = IMAGE_FOLDER + "/" + posts["id"] + ".jpg"

table = posts.copy()
table.insert(0, "thumbnail", sources)

# Stops any < or > in the text from being treated as HTML
for column in ["username", "display_name", "caption"]:
    table[column] = table[column].fillna("").map(html.escape)

HTML(table.to_html(escape=False, formatters={"thumbnail": thumbnail}))

## 4. Reorder by date posted

Oldest first, so that the rows run in the order the posts were made. The index on the left still shows each post's original position in the feed.

In [ ]:
posts_by_date = posts.sort_values("date_posted").copy()
posts_by_date.head()

## 5. Graph post age against position in the feed

The x axis is the order in which the posts appeared in the feed, which is the order of the lines in the NDJSON file (0 is the first post seen). The y axis is each post's age in days when it was captured, from newest at the bottom to oldest at the top.

A purely chronological feed would rise steadily from left to right. Jumps up and down show where older and newer posts are mixed together.

In [ ]:
captured = pd.to_datetime(df["timestamp_collected"].max(), unit="ms")
age_days = (captured - posts["date_posted"]) / pd.Timedelta(days=1)

age_days.plot(title="Post age in days, by position in feed");

## 6. Followed vs recommended posts

In [ ]:
followed = posts[posts["following"]]
recommended = posts[~posts["following"]]

counts = pd.Series({"Followed": len(followed), "Recommended": len(recommended)})
counts.plot.bar(title="Followed vs recommended posts");

## 7. More graphs

### Sponsored vs non-sponsored posts

In [ ]:
posts["paid_partnership"].value_counts().plot.bar(title="Paid partnership posts");

### Average likes: followed vs recommended

In [ ]:
posts.groupby("following")["likes"].mean().plot.bar(title="Average likes by following");

### Media types

In `media_type`, 1 is a static post, 2 is a video and 8 is a carousel.

In [ ]:
posts["media_type"].value_counts().plot.bar(title="Media types");

### Average likes by media type

In [ ]:
posts.groupby("media_type")["likes"].mean().plot.bar(title="Average likes by media type");

### Average likes: sponsored vs non-sponsored

In [ ]:
posts.groupby("paid_partnership")["likes"].mean().plot.bar(title="Average likes by paid partnership");